# Dashboard Prep - Extended Data (2020-2024)

This notebook prepares the extended dataset with all firm classifications (2020-2024) for dashboard use.

It creates summary tables aggregated by:
- Classification Year
- Firm Category
- Industry (NACE Section)
- Region
- Company Age Band

In [ ]:
import pandas as pd
import json
import os

print("Loading extended classifications dataset...")
df = pd.read_pickle('../data/processed/austria_extended_classifications.pkl')
print(f"✓ Loaded: {df.shape[0]:,} companies × {df.shape[1]} variables")

## 1. Category Summary - By Year

In [ ]:
# Create summary of all categories by year
categories = ['scaler', 'hgf', 'consistent_hgf', 'consistent_hypergrower', 
              'gazelle', 'mature_hgf', 'scaleup', 'superstar']

summary_by_year = []

for year in [2020, 2021, 2022, 2023, 2024]:
    row = {'year': year}
    for cat in categories:
        col_name = f'{year}_{cat}'
        count = (df[col_name] == 1).sum()
        row[cat] = count
    summary_by_year.append(row)

summary_df = pd.DataFrame(summary_by_year)
print("\nSummary by Year:")
print(summary_df.to_string())

# Save as CSV and JSON
summary_df.to_csv('../data/processed/classification_summary_by_year.csv', index=False)
summary_df.to_json('../data/processed/classification_summary_by_year.json', orient='records')
print("\n✓ Saved classification_summary_by_year")

## 2. Industry Breakdown - Latest Year (2024)

In [ ]:
# Breakdown by industry for 2024
year_2024 = 2024
industry_summary = []

for industry in df['nace_section'].dropna().unique():
    industry_df = df[df['nace_section'] == industry]
    row = {'industry': industry}
    
    for cat in categories:
        col_name = f'{year_2024}_{cat}'
        count = (industry_df[col_name] == 1).sum()
        row[cat] = count
    
    industry_summary.append(row)

industry_df = pd.DataFrame(industry_summary).sort_values('hgf', ascending=False)
print(f"\nIndustry Breakdown (Year {year_2024}):")
print(industry_df.to_string())

# Save
industry_df.to_csv('../data/processed/classification_summary_by_industry_2024.csv', index=False)
print("\n✓ Saved classification_summary_by_industry_2024")

## 3. Regional Breakdown - Latest Year (2024)

In [ ]:
# Breakdown by region for 2024
region_summary = []

for region in df['region'].dropna().unique():
    region_df = df[df['region'] == region]
    row = {'region': region}
    
    for cat in categories:
        col_name = f'{year_2024}_{cat}'
        count = (region_df[col_name] == 1).sum()
        row[cat] = count
    
    region_summary.append(row)

region_df = pd.DataFrame(region_summary).sort_values('hgf', ascending=False)
print(f"\nRegional Breakdown (Year {year_2024}):")
print(region_df.to_string())

# Save
region_df.to_csv('../data/processed/classification_summary_by_region_2024.csv', index=False)
print("\n✓ Saved classification_summary_by_region_2024")

## 4. Gazelles & Superstars Detail

In [ ]:
# Extract notable firms for each classification
notable_firms = {}

for cat in ['gazelle', 'superstar']:
    col_name = f'{year_2024}_{cat}'
    firms = df[df[col_name] == 1][[
        'company_name', 'region', 'nace_section', 'founded_year',
        f'emp_2021_num', f'emp_2024_num', f'aagr_{year_2024}'
    ]].copy()
    
    # Sort by aagr
    firms[f'aagr_{year_2024}'] = pd.to_numeric(firms[f'aagr_{year_2024}'], errors='coerce')
    firms = firms.sort_values(f'aagr_{year_2024}', ascending=False)
    
    firms.to_csv(f'../data/processed/{cat}s_{year_2024}.csv', index=False)
    notable_firms[cat] = len(firms)
    
    print(f"\nTop 5 {cat.replace('_', ' ').title()}s (Year {year_2024}):")
    print(firms.head(5).to_string())

print(f"\n✓ Saved gazelles and superstars lists")

## 5. Growth Distribution Metrics

In [ ]:
# Calculate growth distribution statistics
growth_stats = {}

for year in [2020, 2021, 2022, 2023, 2024]:
    aagr_col = f'aagr_{year}'
    
    # Convert to numeric
    aagr_numeric = pd.to_numeric(df[aagr_col], errors='coerce')
    
    growth_stats[year] = {
        'count_valid': aagr_numeric.notna().sum(),
        'count_na': (df[aagr_col] == 'n.a.').sum(),
        'mean_aagr': aagr_numeric.mean(),
        'median_aagr': aagr_numeric.median(),
        'min_aagr': aagr_numeric.min(),
        'max_aagr': aagr_numeric.max(),
        'std_aagr': aagr_numeric.std()
    }

growth_stats_df = pd.DataFrame(growth_stats).T
print("\nGrowth Rate Statistics (AAGR):")
print(growth_stats_df.to_string())

growth_stats_df.to_csv('../data/processed/growth_statistics.csv')
print("\n✓ Saved growth_statistics")

## 6. Summary Report

In [ ]:
# Generate summary report
report = f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║              EXTENDED CLASSIFICATIONS - DATA PREPARATION COMPLETE                ║
╚════════════════════════════════════════════════════════════════════════════════╝

DATASET SUMMARY:
├─ Total Companies: {len(df):,}
├─ Original Variables: 29
├─ New Variables: 50
│  ├─ Growth variables (2018-2024): 15
│  ├─ AAGR variables (2020-2024): 5
│  └─ Classification variables (8 × 5 years): 40
└─ Total Variables: {df.shape[1]}

CLASSIFICATION YEARS COVERED: 2020, 2021, 2022, 2023, 2024

FIRM CATEGORIES (8 TYPES):
├─ Growth-Based:
│  ├─ Scalers (AAGR > 10%)
│  ├─ HGFs (AAGR > 20%)
│  ├─ Consistent HGFs (AAGR > 20% + growth > 20% in 2/3 years)
│  └─ Consistent Hypergrowers (AAGR > 20% + growth > 40% in 2/3 years)
└─ Age-Based:
   ├─ Gazelles (Consistent HGFs, age ≤ 10)
   ├─ Mature HGFs (Consistent HGFs, age > 10)
   ├─ Scaleups (Hypergrowers, age ≤ 10)
   └─ Superstars (Hypergrowers, age > 10)

SAVED OUTPUT FILES:
├─ data/processed/classification_summary_by_year.csv
├─ data/processed/classification_summary_by_year.json
├─ data/processed/classification_summary_by_industry_2024.csv
├─ data/processed/classification_summary_by_region_2024.csv
├─ data/processed/gazelles_2024.csv
├─ data/processed/superstars_2024.csv
└─ data/processed/growth_statistics.csv

DASHBOARD ACCESS:
├─ Basic Dashboard: streamlit run app/dashboard.py
└─ Extended Dashboard: streamlit run app/extended_dashboard.py

STATUS: ✓ All data preparation complete
"""

print(report)

# Save report
with open('../data/processed/PREPARATION_REPORT.txt', 'w') as f:
    f.write(report)

print("\n✓ Report saved to data/processed/PREPARATION_REPORT.txt")